# 03. 순수 하이퍼파라미터 효과 측정 — B03 (증강 강제 OFF + Optuna tuned)

02번 노트북의 대조군은 세 개입니다.

```text
B00   증강 강제 OFF    + optimizer auto
B01   YOLO 기본 증강   + optimizer auto
B02   YOLO 기본 증강   + Optuna tuned
```

이 조합만으로는 **순수 하이퍼파라미터 효과**를 볼 수 없습니다.

- `B02 - B00` : 증강 효과와 하이퍼파라미터 효과가 섞여 있습니다.
- `B02 - B01` : 하이퍼파라미터 효과이지만 mosaic이 켜진 조건에서의 값입니다.

증강이 전혀 없는 상태에서의 하이퍼파라미터 효과를 보려면 네 번째 칸이 필요합니다.

|                | optimizer auto | Optuna tuned |
|----------------|----------------|--------------|
| 증강 강제 OFF   | B00            | **B03 (이 노트북)** |
| YOLO 기본 증강  | B01            | B02          |

즉 **`B03 - B00`이 순수 하이퍼파라미터 최적화 효과**입니다.

## 왜 02번과 분리해서 실행하나

02번이 실행 중일 때 같은 `experiment_results.csv`에 동시에 쓰면 결과가 깨질 수 있습니다.
이 노트북은 **자체 폴더에 결과를 저장**하고, 비교할 때만 02번 결과를 읽기 전용으로 불러옵니다.

```text
02번 출력 : ../models/yolo/02_experiment_augmentation/
03번 출력 : ../models/yolo/02_experiment_augmentation/
```

epoch은 02번 최종 단계와 같은 **15**를 사용하므로 그대로 비교할 수 있습니다.


## 1. 라이브러리

In [ ]:
from __future__ import annotations

import json
import math
import time
import random
import hashlib
import shutil
import gc
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml
import torch

from ultralytics import YOLO

try:
    import ultralytics
    ULTRALYTICS_VERSION = ultralytics.__version__
except Exception:
    ULTRALYTICS_VERSION = "unknown"

try:
    from ultralytics.cfg import DEFAULT_CFG_DICT
except Exception:
    DEFAULT_CFG_DICT = {}

from IPython.display import display, Image as IPImage

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 500)


def set_korean_font():
    candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    installed = {font.name for font in fm.fontManager.ttflist}

    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break

    plt.rcParams["axes.unicode_minus"] = False


set_korean_font()

print("Ultralytics:", ULTRALYTICS_VERSION)
print("PyTorch    :", torch.__version__)
print("OpenCV     :", cv2.__version__)

## 2. 경로와 공통 설정

`EPOCHS = 15`, 출력 폴더만 02번과 다릅니다.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# ------------------------------------------------------------
# 1) 1번 노트북이 만든 processed 데이터
# ------------------------------------------------------------
PROCESSED_DIR = (
    PROJECT_ROOT / "../../data/processed"
).resolve()

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"processed 폴더가 없습니다: {PROCESSED_DIR}\n"
        "먼저 01_recycling_eda_preprocess_build_processed.ipynb를 실행하세요."
    )

DATA_YAML = PROCESSED_DIR / "data.yaml"

# ------------------------------------------------------------
# 2) 모델/실험 산출물 위치
# ------------------------------------------------------------
EXPERIMENT_ROOT = (
    PROJECT_ROOT / "../models/yolo/02_experiment_augmentation"
).resolve()

RUNS_DIR = EXPERIMENT_ROOT / "runs"
CV_CACHE_DIR = EXPERIMENT_ROOT / "opencv_datasets"

# ------------------------------------------------------------
# 3) 사람이 확인할 보고서 위치
# ------------------------------------------------------------
REPORT_ROOT = EXPERIMENT_ROOT / "report"
REPORT_SOURCE_DIR = REPORT_ROOT / "preprocess"
SUMMARY_DIR = REPORT_ROOT / "summary"
PER_CLASS_DIR = REPORT_ROOT / "per_class"
FINAL_DIR = REPORT_ROOT / "final_best"

for path in [
    EXPERIMENT_ROOT,
    RUNS_DIR,
    CV_CACHE_DIR,
    REPORT_ROOT,
    REPORT_SOURCE_DIR,
    SUMMARY_DIR,
    PER_CLASS_DIR,
    FINAL_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "yolo26n.pt"
EPOCHS = 15
IMGSZ = 640
BATCH = 8
PATIENCE = 12
# 데이터 로딩을 별도 프로세스로 병렬화합니다 (물리 14코어).
# 46개 런 전부 같은 값으로 돌려야 실험 간 비교가 공정합니다.
WORKERS = 4
SEED = 42

RUN_EXPERIMENTS = True
RUN_MULTI_SEED = False
SKIP_COMPLETED = True
SMOKE_TEST = False

# OpenCV static augmentation은 원본과 같은 데이터 개수를 유지합니다.
CV_APPLY_PROBABILITY = 0.80
CV_OUTPUT_JPEG_QUALITY = 95
CLEANUP_CV_DATASET_AFTER_RUN = True

if SMOKE_TEST:
    EPOCHS = 2
    PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("PROCESSED_DIR  :", PROCESSED_DIR)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("RUNS_DIR       :", RUNS_DIR)
print("REPORT_ROOT    :", REPORT_ROOT)
print("SUMMARY_DIR    :", SUMMARY_DIR)
print("MODEL_NAME     :", MODEL_NAME)
print("EPOCHS         :", EPOCHS)
print("IMGSZ          :", IMGSZ)
print("BATCH          :", BATCH)
print("SMOKE_TEST     :", SMOKE_TEST)


## 3. 01번 Optuna 하이퍼파라미터 불러오기

`selected_trial.json`이 없으면 `optimizer="auto"`로 떨어지는데, 그러면 B03이 B00과 같아져 실험 의미가 없습니다. 아래 출력에서 `HP_SOURCE = optuna_tuned`를 꼭 확인하세요.

In [ ]:
# ============================================================
# 01번 Optuna 결과(최적 하이퍼파라미터) 불러오기
# ============================================================
# selected_trial.json이 있으면 모든 증강 실험에 같은 값을 적용하고,
# 없으면 optimizer="auto" 기본값으로 실행합니다.

USE_OPTUNA_PARAMS = True

# 자동 탐색이 애매하면 경로를 직접 지정하세요.
SELECTED_TRIAL_JSON_OVERRIDE = None

# 01번 Optuna가 탐색한 학습 하이퍼파라미터만 가져옵니다.
TUNABLE_KEYS = [
    "optimizer",
    "lr0",
    "lrf",
    "momentum",
    "weight_decay",
    "warmup_epochs",
    "cos_lr",
]


def find_selected_trial_json():
    if SELECTED_TRIAL_JSON_OVERRIDE is not None:
        path = Path(
            SELECTED_TRIAL_JSON_OVERRIDE
        ).expanduser().resolve()

        if not path.exists():
            raise FileNotFoundError(
                f"지정한 selected_trial.json이 없습니다: {path}"
            )

        return path

    candidates = []

    for root in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        candidates.append(
            root
            / "models"
            / "yolo"
            / "01_yolo_optuna_no_aug"
            / "report"
            / "selected_trial.json"
        )

    for kaggle_root in [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]:
        if kaggle_root.exists():
            candidates.extend(
                sorted(
                    kaggle_root.rglob("selected_trial.json")
                )
            )

    for path in candidates:
        if path.exists():
            return path.resolve()

    return None


TUNED_PARAMS = {}
HP_SOURCE = "auto"
OPTUNA_SELECTED_JSON = None

if not USE_OPTUNA_PARAMS:
    print("USE_OPTUNA_PARAMS=False -> optimizer='auto'로 실행합니다.")

else:
    OPTUNA_SELECTED_JSON = find_selected_trial_json()

    if OPTUNA_SELECTED_JSON is None:
        print("selected_trial.json을 찾지 못했습니다.")
        print("-> optimizer='auto' 기본값으로 실행합니다.")
        print("   01번 Optuna가 끝난 뒤 다시 실행하면 자동으로 적용됩니다.")

    else:
        with open(
            OPTUNA_SELECTED_JSON,
            "r",
            encoding="utf-8",
        ) as file:
            selected_record = json.load(file)

        TUNED_PARAMS = {
            key: value
            for key, value in selected_record.get("params", {}).items()
            if key in TUNABLE_KEYS
        }

        HP_SOURCE = "optuna_tuned"

        print("01번 Optuna 최적 하이퍼파라미터를 불러왔습니다.")
        print("source        :", OPTUNA_SELECTED_JSON)
        print("selected_trial:", selected_record.get("selected_trial"))
        print("trial mAP50-95:", selected_record.get("objective_mAP50_95"))

        for key, value in TUNED_PARAMS.items():
            print(f"  {key:>15} : {value}")

        # 01번과 다른 조건에서 탐색된 값이면 경고만 남깁니다.
        fixed_config = selected_record.get("fixed", {})

        for key, current_value in [
            ("model", MODEL_NAME),
            ("imgsz", IMGSZ),
            ("batch", BATCH),
        ]:
            if (
                key in fixed_config
                and fixed_config[key] != current_value
            ):
                print(
                    f"주의: 01번은 {key}={fixed_config[key]}로 탐색했는데 "
                    f"현재 설정은 {current_value}입니다."
                )

print()
print("HP_SOURCE   :", HP_SOURCE)
print("TUNED_PARAMS:", TUNED_PARAMS)


## 4. GPU 확인

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {total_vram:.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")
else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

## 5. data.yaml 보정

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

dataset_config["path"] = str(PROCESSED_DIR.resolve())

RUNTIME_DATA_YAML = SUMMARY_DIR / "runtime_processed.yaml"

with open(RUNTIME_DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dataset_config, file, allow_unicode=True, sort_keys=False)

names_raw = dataset_config["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(key): str(value) for key, value in names_raw.items()}
else:
    CLASS_NAMES = {index: str(value) for index, value in enumerate(names_raw)}

NUM_CLASSES = len(CLASS_NAMES)

print(RUNTIME_DATA_YAML.read_text(encoding="utf-8")[:5000])
print("클래스 수:", NUM_CLASSES)

## 6. 1번 노트북의 품질 보고서 (있으면)

In [ ]:
CLASS_SUPPORT_CSV = REPORT_SOURCE_DIR / "class_support_processed.csv"
PREPROCESS_SUMMARY_CSV = REPORT_SOURCE_DIR / "preprocess_summary.csv"

if CLASS_SUPPORT_CSV.exists():
    class_support_df = pd.read_csv(CLASS_SUPPORT_CSV)
    display(class_support_df)

    no_val_classes = class_support_df[class_support_df["val"].eq(0)]
    print("Validation object가 0인 클래스:", len(no_val_classes))
    if len(no_val_classes):
        display(no_val_classes[["class_name", "train", "val"]])
else:
    class_support_df = pd.DataFrame()
    print("class_support_processed.csv를 찾지 못했습니다.")

if PREPROCESS_SUMMARY_CSV.exists():
    display(pd.read_csv(PREPROCESS_SUMMARY_CSV))

## 7. processed 데이터 빠른 확인

In [ ]:
TRAIN_IMAGE_DIR = PROCESSED_DIR / "images" / "train"
VAL_IMAGE_DIR = PROCESSED_DIR / "images" / "val"
TRAIN_LABEL_DIR = PROCESSED_DIR / "labels" / "train"
VAL_LABEL_DIR = PROCESSED_DIR / "labels" / "val"

for path in [TRAIN_IMAGE_DIR, VAL_IMAGE_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)

train_images = sorted(path for path in TRAIN_IMAGE_DIR.iterdir() if path.is_file())
val_images = sorted(path for path in VAL_IMAGE_DIR.iterdir() if path.is_file())
train_labels = sorted(TRAIN_LABEL_DIR.glob("*.txt"))
val_labels = sorted(VAL_LABEL_DIR.glob("*.txt"))

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))
print("Val images  :", len(val_images))
print("Val labels  :", len(val_labels))

if len(train_images) != len(train_labels):
    raise ValueError("Train image와 label 개수가 다릅니다.")

if len(val_images) != len(val_labels):
    raise ValueError("Validation image와 label 개수가 다릅니다.")

if len(val_images) == 0:
    raise ValueError("Validation 이미지가 없습니다.")

print("Processed quick check: PASSED")

## 8. 증강 설정 (전부 0)

In [ ]:
AUGMENTATION_KEYS = [
    "hsv_h", "hsv_s", "hsv_v",
    "degrees", "translate", "scale", "shear", "perspective",
    "flipud", "fliplr", "bgr",
    "mosaic", "mixup", "cutmix", "copy_paste",
    "close_mosaic", "augmentations",
]

installed_aug_defaults = {
    key: DEFAULT_CFG_DICT.get(key, "<not available>")
    for key in AUGMENTATION_KEYS
}

display(
    pd.DataFrame({
        "argument": list(installed_aug_defaults.keys()),
        "installed_default": list(installed_aug_defaults.values()),
    })
)

In [ ]:
NO_AUG = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 0,
    "augmentations": [],
}


def with_no_aug(**changes):
    config = dict(NO_AUG)
    config.update(changes)
    return config


def supported_train_args(config: dict | None):
    """현재 Ultralytics 버전에서 지원되는 key만 남깁니다."""
    if config is None:
        return {}, []

    if not DEFAULT_CFG_DICT:
        # config dictionary를 가져오지 못한 버전에서는 그대로 전달합니다.
        return dict(config), []

    supported = set(DEFAULT_CFG_DICT.keys())
    unknown = sorted(set(config.keys()) - supported)
    filtered = {key: value for key, value in config.items() if key in supported}

    return filtered, unknown

## 9. B03 실험 정의

`aug`는 `NO_AUG`(모든 증강 인자를 0으로 강제)이고 `hp_mode`는 `tuned`입니다.
02번의 B00과 **증강 조건이 완전히 동일**하고 하이퍼파라미터만 다릅니다.

In [ ]:
EXPERIMENTS = [
    {
        "id": "B03",
        "name": "no_aug_tuned",
        "family": "baseline",
        "data_kind": "processed",
        "cv_policy": None,
        "aug": NO_AUG,
        "required_args": [],
        "hp_mode": "tuned",
        "description": "증강 강제 OFF + Optuna tuned (순수 하이퍼파라미터 효과)",
    },
]

for experiment in EXPERIMENTS:
    experiment.setdefault("hp_mode", "tuned")

display(pd.DataFrame(EXPERIMENTS)[["id", "name", "hp_mode", "description"]])


## 10. 실행에 필요한 함수들

02번과 완전히 같은 코드를 사용합니다.

In [ ]:
def missing_required_args(experiment):
    if not DEFAULT_CFG_DICT:
        return []

    supported = set(DEFAULT_CFG_DICT.keys())
    return [
        argument
        for argument in experiment.get("required_args", [])
        if argument not in supported
    ]


compatibility_rows = []

for experiment in EXPERIMENTS:
    missing = missing_required_args(experiment)
    compatibility_rows.append({
        "id": experiment["id"],
        "name": experiment["name"],
        "supported": len(missing) == 0,
        "missing_required_args": ", ".join(missing),
    })

compatibility_df = pd.DataFrame(compatibility_rows)
display(compatibility_df)

In [ ]:
def dataset_yaml_for_experiment(experiment, seed):
    # 이 노트북은 processed 원본만 사용합니다.
    if experiment["data_kind"] != "processed":
        raise ValueError(
            f"이 노트북은 processed 데이터만 지원합니다: {experiment['data_kind']}"
        )

    return RUNTIME_DATA_YAML


In [ ]:
def extract_metrics(metrics):
    box = metrics.box

    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

In [ ]:
def save_per_class_metrics(metrics, experiment_id, experiment_name, seed):
    maps = np.asarray(metrics.box.maps, dtype=float)

    per_class = pd.DataFrame({
        "class_id": range(len(maps)),
        "class_name": [CLASS_NAMES.get(index, f"class_{index}") for index in range(len(maps))],
        "mAP50_95": maps,
    })

    if len(class_support_df):
        support_cols = [col for col in ["class_name", "train", "val"] if col in class_support_df.columns]
        per_class = per_class.merge(
            class_support_df[support_cols],
            on="class_name",
            how="left",
        )

    path = PER_CLASS_DIR / f"{experiment_id}_{experiment_name}_seed{seed}.csv"
    per_class.to_csv(path, index=False, encoding="utf-8-sig")
    return path

In [ ]:
RESULTS_CSV = SUMMARY_DIR / "experiment_results.csv"


def hp_source_for(experiment):
    """이 실험이 실제로 어떤 하이퍼파라미터로 학습되는지 반환합니다."""
    if experiment.get("hp_mode", "tuned") == "tuned" and TUNED_PARAMS:
        return "optuna_tuned"

    return "auto"


def train_one_experiment(experiment, seed=SEED, epochs=None):
    epochs = EPOCHS if epochs is None else epochs

    missing = missing_required_args(experiment)

    if missing:
        return {
            "status": "SKIPPED_UNSUPPORTED",
            "id": experiment["id"],
            "name": experiment["name"],
            "family": experiment["family"],
            "seed": seed,
            "requested_epochs": epochs,
            "hp_mode": experiment.get("hp_mode", "tuned"),
            "hp_source": hp_source_for(experiment),
            "error": f"unsupported args: {missing}",
        }

    dataset_yaml = dataset_yaml_for_experiment(experiment, seed)
    # epoch 수가 다르면 서로 덮어쓰지 않도록 run 이름을 구분합니다.
    run_name = f"{experiment['id']}_{experiment['name']}_seed{seed}_e{epochs}"

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 매우 중요: 모든 실험을 같은 pretrained weight에서 새로 시작합니다.
    model = YOLO(MODEL_NAME)

    train_kwargs = {
        "data": str(dataset_yaml),
        "epochs": epochs,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "patience": PATIENCE,
        "device": DEVICE,
        "workers": WORKERS,
        "seed": seed,
        "deterministic": True,
        # TUNED_PARAMS가 있으면 아래에서 덮어씁니다.
        "optimizer": "auto",
        "amp": True,
        "cache": False,
        "project": str(RUNS_DIR),
        "name": run_name,
        "exist_ok": True,
        "plots": True,
        "verbose": True,
    }

    # hp_mode가 "tuned"인 실험에만 01번 Optuna 결과를 적용합니다.
    # B00 / B01은 "auto"이므로 위의 optimizer="auto" 기본값을 그대로 씁니다.
    experiment_hp_source = hp_source_for(experiment)

    if experiment_hp_source == "optuna_tuned":
        train_kwargs.update(TUNED_PARAMS)

    filtered_aug, unknown_aug = supported_train_args(experiment["aug"])

    if experiment["aug"] is not None:
        train_kwargs.update(filtered_aug)

    start_time = time.perf_counter()
    train_result = model.train(**train_kwargs)
    train_minutes = (time.perf_counter() - start_time) / 60.0

    save_dir = Path(train_result.save_dir)
    best_pt = save_dir / "weights" / "best.pt"

    if not best_pt.exists():
        raise FileNotFoundError(best_pt)

    history_csv = save_dir / "results.csv"
    actual_epochs = np.nan

    if history_csv.exists():
        history = pd.read_csv(history_csv)
        actual_epochs = len(history)

    best_model = YOLO(str(best_pt))

    val_metrics = best_model.val(
        data=str(dataset_yaml),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=False,
        verbose=False,
    )

    per_class_path = save_per_class_metrics(
        val_metrics,
        experiment["id"],
        experiment["name"],
        seed,
    )

    row = {
        "status": "OK",
        "id": experiment["id"],
        "name": experiment["name"],
        "family": experiment["family"],
        "description": experiment["description"],
        "seed": seed,
        "data_kind": experiment["data_kind"],
        "cv_policy": experiment["cv_policy"],
        "dataset_yaml": str(dataset_yaml),
        "requested_epochs": epochs,
        "actual_epochs": actual_epochs,
        "train_minutes": train_minutes,
        "best_pt": str(best_pt),
        "save_dir": str(save_dir),
        "per_class_csv": str(per_class_path),
        "unknown_filtered_aug_args": ",".join(unknown_aug),
        "hp_mode": experiment.get("hp_mode", "tuned"),
        "hp_source": experiment_hp_source,
        "hp_params": json.dumps(
            TUNED_PARAMS if experiment_hp_source == "optuna_tuned" else {},
            ensure_ascii=False,
        ),
        **extract_metrics(val_metrics),
    }

    # OpenCV static dataset은 용량이 클 수 있으므로 결과 보고서를 보존한 뒤 cache를 정리합니다.
    if experiment["data_kind"] == "opencv" and CLEANUP_CV_DATASET_AFTER_RUN:
        dataset_dir = Path(dataset_yaml).parent
        generation_report = dataset_dir / "generation_report.csv"

        if generation_report.exists():
            persistent_report = SUMMARY_DIR / f"opencv_generation_{experiment['cv_policy']}_seed{seed}.csv"
            shutil.copy2(generation_report, persistent_report)

        shutil.rmtree(dataset_dir, ignore_errors=True)

    del model
    del best_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

In [ ]:
def load_existing_results():
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def has_successful_run(existing_df, experiment, seed, epochs):
    """같은 (실험, seed, epoch, 하이퍼파라미터 설정) 조합이 이미 성공했는지 확인합니다."""
    if not len(existing_df):
        return False

    required_cols = {"status", "id", "seed"}
    if not required_cols.issubset(existing_df.columns):
        return False

    mask = (
        existing_df["status"].eq("OK")
        & existing_df["id"].eq(experiment["id"])
        & existing_df["seed"].eq(seed)
    )

    # epoch이 다르면 다른 실험입니다. (10 epoch 스크리닝 vs 15 epoch 최종 비교)
    if "requested_epochs" in existing_df.columns:
        mask = mask & existing_df["requested_epochs"].eq(epochs)
    else:
        return False

    # 하이퍼파라미터 설정이 바뀌면(auto <-> optuna_tuned) 같은 표에 둘 수 없으므로
    # 건너뛰지 않고 다시 학습합니다.
    if "hp_source" in existing_df.columns:
        previous_source = existing_df["hp_source"].fillna("auto")
    else:
        previous_source = pd.Series("auto", index=existing_df.index)

    mask = mask & previous_source.eq(hp_source_for(experiment))

    return bool(mask.any())


def run_experiment_list(experiments, seed=SEED, epochs=None):
    epochs = EPOCHS if epochs is None else epochs

    existing_df = load_existing_results()
    new_rows = []

    for index, experiment in enumerate(experiments, start=1):
        print("=" * 100)
        print(
            f"[{index}/{len(experiments)}] {experiment['id']} - {experiment['name']}"
            f" - seed={seed} - {epochs} epoch - {hp_source_for(experiment)}"
        )
        print(experiment["description"])

        if SKIP_COMPLETED and has_successful_run(existing_df, experiment, seed, epochs):
            print("이미 성공한 결과가 있어 건너뜁니다.")
            continue

        try:
            row = train_one_experiment(experiment, seed=seed, epochs=epochs)
        except Exception as error:
            row = {
                "status": "FAILED",
                "id": experiment["id"],
                "name": experiment["name"],
                "family": experiment["family"],
                "seed": seed,
                "requested_epochs": epochs,
                "hp_mode": experiment.get("hp_mode", "tuned"),
                "hp_source": hp_source_for(experiment),
                "error": repr(error),
            }

        new_rows.append(row)

        current_existing = load_existing_results()
        combined = pd.concat(
            [current_existing, pd.DataFrame([row])],
            ignore_index=True,
        )

        # 같은 id+seed가 여러 번 존재하면 가장 최근 행을 유지합니다.
        dedup_keys = [
            key
            for key in ["id", "seed", "requested_epochs", "hp_source"]
            if key in combined.columns
        ]

        if dedup_keys:
            combined = combined.drop_duplicates(
                subset=dedup_keys,
                keep="last",
            )

        combined.to_csv(
            RESULTS_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        display(pd.DataFrame([row]))

    return load_existing_results()

## 11. B03 학습 실행

15 epoch 한 번이라 약 1.5시간 예상입니다.

In [ ]:
if RUN_EXPERIMENTS:
    b03_results_df = run_experiment_list(
        EXPERIMENTS,
        seed=SEED,
        epochs=EPOCHS,
    )

else:
    print("RUN_EXPERIMENTS=False")
    b03_results_df = load_existing_results()

display(b03_results_df)


## 12. 02번 결과와 합쳐 2x2 비교

02번의 `experiment_results.csv`에서 **15 epoch** B00 / B01 / B02를 읽어옵니다.
02번 최종 단계가 아직 끝나지 않았다면 그 행이 없을 수 있는데,
그때는 있는 것만 비교하고 안내를 출력합니다.

In [ ]:
# 02번 결과를 읽기 전용으로 참조합니다. (02번이 실행 중이어도 안전합니다)
NB02_SUMMARY_DIR = (
    PROJECT_ROOT
    / "../models/yolo/02_experiment_augmentation/report/summary"
).resolve()

print("02번 결과 폴더:", NB02_SUMMARY_DIR)
print("존재 여부      :", NB02_SUMMARY_DIR.exists())


In [ ]:
set_korean_font()

combined_rows = []

nb02_results_path = NB02_SUMMARY_DIR / "experiment_results.csv"

if nb02_results_path.exists():
    nb02_df = pd.read_csv(nb02_results_path)

    nb02_df = nb02_df[
        nb02_df["status"].eq("OK")
        & nb02_df["requested_epochs"].eq(EPOCHS)
        & nb02_df["id"].isin(["B00", "B01", "B02"])
    ]

    combined_rows.append(nb02_df)

    missing = {"B00", "B01", "B02"} - set(nb02_df["id"])

    if missing:
        print(
            f"02번에서 아직 {EPOCHS} epoch 결과가 없는 실험:",
            sorted(missing),
        )

else:
    print("02번 experiment_results.csv를 찾지 못했습니다.")

own_df = load_existing_results()

own_df = own_df[
    own_df["status"].eq("OK")
    & own_df["requested_epochs"].eq(EPOCHS)
]

combined_rows.append(own_df)

compare_df = pd.concat(combined_rows, ignore_index=True)

LABELS = {
    "B00": "B00  증강OFF + auto",
    "B01": "B01  기본증강 + auto",
    "B02": "B02  기본증강 + tuned",
    "B03": "B03  증강OFF + tuned",
}

compare_df["설명"] = compare_df["id"].map(LABELS)

order = [key for key in ["B00", "B01", "B02", "B03"] if key in set(compare_df["id"])]

compare_df = (
    compare_df
    .drop_duplicates(subset=["id"], keep="last")
    .set_index("id")
    .loc[order]
    .reset_index()
)

display(
    compare_df[
        ["id", "설명", "hp_source", "precision", "recall", "f1", "mAP50", "mAP50_95"]
    ].round(4)
)

COMPARE_CSV = SUMMARY_DIR / "hyperparameter_effect_2x2.csv"
compare_df.to_csv(COMPARE_CSV, index=False, encoding="utf-8-sig")
print("Saved:", COMPARE_CSV)


## 13. 효과 분해 그래프

In [ ]:
set_korean_font()

values = dict(zip(compare_df["id"], compare_df["mAP50_95"]))

# ------------------------------------------------------------
# 1) 네 설정 막대그래프
# ------------------------------------------------------------
plt.figure(figsize=(10, 5.5))

colors = ["#90a4ae", "#90a4ae", "#1565c0", "#1565c0"]

bars = plt.bar(
    compare_df["설명"],
    compare_df["mAP50_95"],
    color=colors[: len(compare_df)],
)

for bar, value in zip(bars, compare_df["mAP50_95"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom",
    )

plt.ylabel("Validation mAP50-95")
plt.title(f"증강 x 하이퍼파라미터 2x2 비교 ({EPOCHS} epoch)")
plt.xticks(rotation=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

figure_path = FINAL_DIR / "hyperparameter_effect_2x2.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)


# ------------------------------------------------------------
# 2) 효과 분해
# ------------------------------------------------------------
effects = []

if {"B00", "B03"} <= values.keys():
    effects.append(("순수 하이퍼파라미터 효과\n(B03 - B00, 증강 없음)", values["B03"] - values["B00"]))

if {"B01", "B02"} <= values.keys():
    effects.append(("하이퍼파라미터 효과\n(B02 - B01, 기본증강)", values["B02"] - values["B01"]))

if {"B00", "B01"} <= values.keys():
    effects.append(("증강 효과\n(B01 - B00, auto)", values["B01"] - values["B00"]))

if {"B03", "B02"} <= values.keys():
    effects.append(("증강 효과\n(B02 - B03, tuned)", values["B02"] - values["B03"]))

if effects:
    names = [name for name, _ in effects]
    deltas = [delta for _, delta in effects]

    plt.figure(figsize=(10, 5.5))

    bars = plt.barh(
        names,
        deltas,
        color=["#2e7d32" if d > 0 else "#c62828" for d in deltas],
    )

    for bar, delta in zip(bars, deltas):
        plt.text(
            delta,
            bar.get_y() + bar.get_height() / 2,
            f" {delta:+.4f}",
            va="center",
        )

    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("mAP50-95 변화량")
    plt.title("효과 분해: 증강과 하이퍼파라미터를 각각 분리")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    figure_path = FINAL_DIR / "effect_decomposition.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)

else:
    print("비교할 조합이 부족합니다. 02번 15 epoch 결과가 나온 뒤 다시 실행하세요.")


## 14. 해석 요령

- **`B03 - B00` 이 거의 0이라면** 01번 Optuna 하이퍼파라미터가 17개 카테고리에서는
  이득을 주지 못한다는 뜻입니다. 86개 카테고리에서 탐색한 값이라 충분히 가능한 결과이고,
  이 경우 17개 기준으로 Optuna를 다시 돌릴 가치가 있습니다.
- **`B03 - B00`은 큰데 `B02 - B01`이 작다면** 증강이 켜진 상태에서는 튜닝 효과가
  묻힌다는 뜻입니다. 증강과 하이퍼파라미터가 상호작용하는 전형적인 모습입니다.
- 차이가 **±0.005~0.02** 수준이면 seed 하나 차이로도 나올 수 있는 범위입니다.
  확정하려면 seed를 바꿔 몇 번 더 돌려봐야 합니다.
